# CodonTransformer fine-tuning smoke test on Google Colab

This notebook keeps code in GitHub, downloads the immutable pretrained model from Hugging Face, reads cluster-cleaned training JSONL from Google Drive, and writes checkpoints/logs/configuration back to Drive. Colab local storage is temporary and is not used for persistent checkpoints.

## 1. Check the NVIDIA GPU
Select a GPU runtime before continuing: **Runtime → Change runtime type → GPU**.

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "CUDA GPU is required for this notebook"
print("CUDA device:", torch.cuda.get_device_name(0))

## 2. Configure repository, model, data, and output locations
The repository is private by default. Add a fine-grained read-only token as the Colab secret `GITHUB_TOKEN`; never paste it into this notebook. All paths used later derive from this configuration cell.

In [ ]:
import os
from pathlib import Path

REPO_URL = os.environ.get(
    "GITHUB_REPO_URL",
    "https://github.com/Yuano0o/codontransformer-nb.git",
)
COLAB_WORKSPACE = Path(os.environ.get("COLAB_WORKSPACE", "/content"))
PROJECT_DIR = COLAB_WORKSPACE / "codontransformer-project"
UPSTREAM_REPO_URL = "https://github.com/Adibvafa/CodonTransformer.git"
UPSTREAM_COMMIT = "4a447b01dab860feb81b647ff1ff88ad598517f4"
HF_MODEL_ID = "adibvafa/CodonTransformer"
DRIVE_MOUNT = Path(os.environ.get("COLAB_DRIVE_MOUNT", "/content/drive"))
DRIVE_PROJECT_SUBDIR = Path("MyDrive") / "CodonTransformer"
DRIVE_TRAINING_DATA = Path("data") / "stage2" / "final_csi_cohorts" / "experiments" / "csi_top10_hc" / "train.jsonl"
SMOKE_SAMPLE_COUNT = 8
SMOKE_STEPS = 8
SMOKE_SEED = 23
RUN_NAME = os.environ.get("COLAB_RUN_NAME", "smoke_test_cuda_csi_top10_8steps")

assert REPO_URL.startswith("https://github.com/")
assert 5 <= SMOKE_STEPS <= 10
assert SMOKE_SAMPLE_COUNT == SMOKE_STEPS

## 3. Clone the GitHub project and pinned upstream source

In [ ]:
import subprocess
import tempfile
from google.colab import userdata

if not (PROJECT_DIR / ".git").is_dir():
    try:
        github_token = userdata.get("GITHUB_TOKEN")
    except Exception:
        raise RuntimeError("Colab secret GITHUB_TOKEN is unavailable") from None
    if not github_token:
        raise RuntimeError("Colab secret GITHUB_TOKEN is empty")
    with tempfile.TemporaryDirectory(prefix="git-askpass-") as askpass_dir:
        askpass_path = Path(askpass_dir) / "askpass.sh"
        askpass_path.write_text(
            '#!/bin/sh\n'
            'case "$1" in\n'
            '  *sername*) printf \'%s\\n\' \'x-access-token\' ;;\n'
            '  *assword*) printf \'%s\\n\' "$COLAB_GITHUB_TOKEN" ;;\n'
            '  *) exit 1 ;;\n'
            'esac\n',
            encoding="utf-8",
        )
        askpass_path.chmod(0o700)
        clone_env = os.environ.copy()
        clone_env.update({
            "GIT_ASKPASS": str(askpass_path),
            "GIT_TERMINAL_PROMPT": "0",
            "COLAB_GITHUB_TOKEN": github_token,
        })
        try:
            subprocess.run(
                ["git", "clone", REPO_URL, str(PROJECT_DIR)],
                check=True,
                env=clone_env,
            )
        finally:
            clone_env.pop("COLAB_GITHUB_TOKEN", None)
            github_token = None
os.chdir(PROJECT_DIR)

upstream_dir = PROJECT_DIR / "upstream"
if not (upstream_dir / ".git").is_dir():
    subprocess.run(
        ["git", "clone", UPSTREAM_REPO_URL, str(upstream_dir)],
        check=True,
    )
subprocess.run(
    ["git", "-C", str(upstream_dir), "checkout", UPSTREAM_COMMIT],
    check=True,
)
print("Project:", PROJECT_DIR)
print("Upstream commit:", UPSTREAM_COMMIT)

## 4. Install Colab dependencies
The requirements file deliberately leaves Colab's CUDA-enabled PyTorch installation unchanged. The pinned upstream source is imported directly from the cloned checkout, avoiding an editable-build dependency on Colab's Python packaging stack.

In [ ]:
subprocess.run(
    [os.sys.executable, "-m", "pip", "install", "-r", "requirements-colab.txt"],
    check=True,
)
UPSTREAM_DIR = (PROJECT_DIR / "upstream").resolve()
assert (UPSTREAM_DIR / "CodonTransformer").is_dir()
upstream_pythonpath = str(UPSTREAM_DIR)
existing_pythonpath = os.environ.get("PYTHONPATH", "")
os.environ["PYTHONPATH"] = os.pathsep.join(
    value for value in (upstream_pythonpath, existing_pythonpath) if value
)
if upstream_pythonpath not in os.sys.path:
    os.sys.path.insert(0, upstream_pythonpath)
import CodonTransformer
assert Path(CodonTransformer.__file__).resolve().is_relative_to(UPSTREAM_DIR)
print("Using pinned upstream source:", CodonTransformer.__file__)

## 5. Mount Google Drive and create the deterministic smoke subset
The source JSONL is the `csi_top10_hc` cluster-aware split in the actual upstream input format (`codons` string plus integer `organism`). The notebook selects eight records with a fixed seed into temporary Colab storage; the original Drive dataset is never changed.

In [ ]:
import random
from google.colab import drive

drive.mount(str(DRIVE_MOUNT))
DRIVE_ROOT = DRIVE_MOUNT / DRIVE_PROJECT_SUBDIR
DATASET_PATH = DRIVE_ROOT / DRIVE_TRAINING_DATA
RUN_DIR = DRIVE_ROOT / "runs" / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
assert DATASET_PATH.is_file(), f"Training JSONL not found: {DATASET_PATH}"
all_records = [line for line in DATASET_PATH.read_text(encoding="utf-8").splitlines() if line.strip()]
assert len(all_records) >= SMOKE_SAMPLE_COUNT, f"Need at least {SMOKE_SAMPLE_COUNT} records"
smoke_records = random.Random(SMOKE_SEED).sample(all_records, SMOKE_SAMPLE_COUNT)
SMOKE_DATASET_PATH = PROJECT_DIR / "runtime" / f"csi_top10_hc_smoke_{SMOKE_SAMPLE_COUNT}.jsonl"
SMOKE_DATASET_PATH.parent.mkdir(parents=True, exist_ok=True)
SMOKE_DATASET_PATH.write_text("\n".join(smoke_records) + "\n", encoding="utf-8")
assert sum(1 for _ in SMOKE_DATASET_PATH.open(encoding="utf-8")) == SMOKE_SAMPLE_COUNT
print("Source training data:", DATASET_PATH)
print("Temporary smoke subset:", SMOKE_DATASET_PATH)
print("Persistent run directory:", RUN_DIR)

## 6. Download the pretrained model from Hugging Face
The model is downloaded only to Colab temporary storage. It is never committed to GitHub.

In [ ]:
MODEL_DIR = PROJECT_DIR / "models" / "pretrained"
subprocess.run(
    [
        os.sys.executable,
        "scripts/download_pretrained.py",
        "--repo-id",
        HF_MODEL_ID,
        "--output-dir",
        str(MODEL_DIR),
    ],
    check=True,
)
assert (MODEL_DIR / "model.safetensors").is_file()
print("Temporary model directory:", MODEL_DIR)

## 7. Run the CUDA smoke test
The notebook runs exactly eight optimizer steps over the eight-record `csi_top10_hc` subset. This is a CUDA environment check, not formal fine-tuning. The output directory is on Drive, so checkpoints, Lightning CSV logs, `training.log`, `resolved_config.yaml`, and `runtime.json` persist after Colab disconnects.

In [ ]:
subprocess.run(
    [
        os.sys.executable,
        "scripts/finetune_codontransformer.py",
        "--config",
        "configs/smoke_test_cuda_csi_top10.yaml",
        "--model-dir",
        str(MODEL_DIR),
        "--dataset-path",
        str(SMOKE_DATASET_PATH),
        "--output-dir",
        str(RUN_DIR),
        "--limit-train-batches",
        str(SMOKE_STEPS),
    ],
    check=True,
)
CHECKPOINT_PATH = RUN_DIR / "checkpoints" / "last.ckpt"
assert CHECKPOINT_PATH.is_file(), f"Checkpoint not found: {CHECKPOINT_PATH}"
print("Checkpoint saved to Drive:", CHECKPOINT_PATH)

## 8. Reload the checkpoint and verify inference by translation

In [ ]:
VALIDATION_OUTPUT = RUN_DIR / "checkpoint_inference_validation.json"
subprocess.run(
    [
        os.sys.executable,
        "scripts/validate_checkpoint_inference.py",
        "--model-dir",
        str(MODEL_DIR),
        "--checkpoint",
        str(CHECKPOINT_PATH),
        "--output",
        str(VALIDATION_OUTPUT),
        "--device",
        "cuda",
        "--organism",
        "Nicotiana tabacum",
    ],
    check=True,
)
import json
validation = json.loads(VALIDATION_OUTPUT.read_text())
assert validation["translation_verified"] is True
validation

## 9. Inspect persistent Drive outputs
Do not end the session until `last.ckpt` and the validation JSON are visible under the Drive run directory.

In [ ]:
for path in sorted(RUN_DIR.rglob("*")):
    if path.is_file():
        print(path.relative_to(RUN_DIR), path.stat().st_size)